In [ ]:
### Cell 1 — Imports & Load Data

import sys, os, json
sys.path.append('../src')
from preprocessing import load_and_preprocess
from gb_model import train_gb, evaluate, cross_validate_gb

import numpy as np
import matplotlib.pyplot as plt

os.makedirs('../report/figures', exist_ok=True)

X_train, X_test, y_train, y_test, feature_names = load_and_preprocess()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features: {feature_names}')
print(f'Baseline to beat → MAE=0.1079  RMSE=0.1326')

In [ ]:
### Cell 2 — Tune n_estimators

print("5-fold CV: tuning n_estimators...\n")
est_results = []
for n_est in [50, 100, 200, 300, 500]:
    mean_mae, std_mae = cross_validate_gb(X_train, y_train, n_estimators=n_est, cv=5)
    est_results.append((n_est, mean_mae, std_mae))
    print(f"  n_estimators={n_est:>3} | CV MAE = {mean_mae:.4f} ± {std_mae:.4f}")

best_n = min(est_results, key=lambda x: x[1])[0]
print(f"\n✅ Best n_estimators: {best_n}")


In [ ]:
### Cell 3 — Tune learning_rate

print(f"\nTuning learning_rate (n_estimators={best_n})...\n")
lr_results = []
for lr in [0.01, 0.05, 0.1, 0.2]:
    mean_mae, std_mae = cross_validate_gb(X_train, y_train,
                                          n_estimators=best_n, learning_rate=lr, cv=5)
    lr_results.append((lr, mean_mae))
    print(f"  learning_rate={lr} | CV MAE = {mean_mae:.4f}")

best_lr = min(lr_results, key=lambda x: x[1])[0]
print(f"\n✅ Best learning_rate: {best_lr}")

In [ ]:
### Cell 4 — Train Final Model

gb_model = train_gb(X_train, y_train,
                    n_estimators=best_n,
                    learning_rate=best_lr)
metrics = evaluate(gb_model, X_test, y_test)
y_pred_gb = metrics['y_pred']

print('=' * 55)
print('  GRADIENT BOOSTING — FINAL TEST RESULTS')
print('=' * 55)
print(f"  MAE  : {metrics['mae']:.4f}  (baseline: 0.1079)")
print(f"  RMSE : {metrics['rmse']:.4f}  (baseline: 0.1326)")
improvement = (0.1079 - metrics['mae']) / 0.1079 * 100
print(f"  MAE improvement over baseline: {improvement:.1f}%")
if metrics['mae'] < 0.1079:
    print("  ✅ Beat the baseline!")
else:
    print("  ⚠️  Try smaller max_depth or more trees")
print('=' * 55)

In [ ]:
### Cell 5 — Feature Importance Chart

importances = gb_model.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(11, 5))
plt.bar(range(len(indices)), importances[indices],
        color='darkorange', edgecolor='black')
plt.xticks(range(len(indices)),
           [feature_names[i] for i in indices],
           rotation=45, ha='right', fontsize=9)
plt.title('Gradient Boosting — Top 15 Feature Importances\n'
          f'(n_estimators={best_n}, lr={best_lr})', fontsize=12)
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('../report/figures/gb_feature_importance.png', dpi=150)
plt.show()
print('Saved → report/figures/gb_feature_importance.png')

print('\nTop 5 features:')
for i in indices[:5]:
    print(f'  {feature_names[i]:<25} {importances[i]:.4f}')

In [ ]:
### Cell 6 — Predicted vs Actual Scatter

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_gb, alpha=0.5, s=20,
            color='darkorange', edgecolors='white', linewidths=0.3)
lims = [min(y_test.min(), y_pred_gb.min()) - 0.1,
        max(y_test.max(), y_pred_gb.max()) + 0.1]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Health Risk Score')
plt.ylabel('Predicted Health Risk Score')
plt.title(f'Gradient Boosting: Predicted vs Actual\n'
          f'MAE={metrics["mae"]:.4f}   RMSE={metrics["rmse"]:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('../report/figures/gb_scatter.png', dpi=150)
plt.show()
print('Saved → report/figures/gb_scatter.png')


In [ ]:
### Cell 7 — Time-Series Plot

plt.figure(figsize=(14, 4))
plt.plot(y_test,    label='Actual', color='black',      linewidth=1.5)
plt.plot(y_pred_gb, label='GB Predicted', color='darkorange',
         linewidth=1.2, alpha=0.85)
plt.title('Gradient Boosting — Test Set Predictions (200 records)')
plt.xlabel('Test Record Index')
plt.ylabel('Health Risk Score')
plt.legend()
plt.tight_layout()
plt.savefig('../report/figures/gb_timeseries.png', dpi=150)
plt.show()
print('Saved → report/figures/gb_timeseries.png')

In [ ]:
### Cell 8 — Save Results JSON

gb_results = {
    'model': 'Gradient Boosting',
    'best_n_estimators': int(best_n),
    'best_learning_rate': float(best_lr),
    'mae':  metrics['mae'],
    'rmse': metrics['rmse'],
    'top_5_features': [feature_names[i] for i in indices[:5]]
}

with open('../report/gb_results.json', 'w') as f:
    json.dump(gb_results, f, indent=2)

print('✅ Saved → report/gb_results.json')
print(gb_results)